In [1]:
# -- Import libraries
import os
import time
import re
import pickle
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")


In [2]:
# -- Configuration
BASE_URL = "https://www.wrestlestat.com"
DUALS_URL = "https://www.wrestlestat.com/d1/event/recentduals"

# File paths
DATA_DIR = Path("wrestling_data")
DATA_DIR.mkdir(exist_ok=True)

LINKS_FILE = DATA_DIR / "seen_dual_links.pkl"
WRESTLERS_FILE = DATA_DIR / "wrestlers.csv"
TEAMS_FILE = DATA_DIR / "teams.csv"
DUAL_MEETS_FILE = DATA_DIR / "dual_meets.csv"
MATCHES_FILE = DATA_DIR / "matches.csv"

In [3]:
def load_or_create_df(filepath, columns):
    """Load existing CSV or create new DataFrame"""
    if filepath.exists():
        return pd.read_csv(filepath)
    return pd.DataFrame(columns=columns)

def get_or_create_wrestler(wrestlers_df, name):
    """Get wrestler_id or create new wrestler entry - MODIFIES wrestlers_df in place"""
    if name is None or pd.isna(name):
        return None, wrestlers_df
    
    name = name.strip()
    existing = wrestlers_df[wrestlers_df['name'] == name]
    
    if not existing.empty:
        return existing.iloc[0]['wrestler_id'], wrestlers_df
    
    # Create new wrestler
    new_id = int(wrestlers_df['wrestler_id'].max()) + 1 if len(wrestlers_df) > 0 else 1
    new_wrestler = pd.DataFrame({
        'wrestler_id': [new_id],
        'name': [name]
    })
    wrestlers_df = pd.concat([wrestlers_df, new_wrestler], ignore_index=True)
    return new_id, wrestlers_df

def get_or_create_team(teams_df, name):
    """Get team_id or create new team entry - MODIFIES teams_df in place"""
    if name is None or pd.isna(name):
        return None, teams_df
    
    name = name.strip()
    existing = teams_df[teams_df['name'] == name]
    
    if not existing.empty:
        return existing.iloc[0]['team_id'], teams_df
    
    # Create new team
    new_id = int(teams_df['team_id'].max()) + 1 if len(teams_df) > 0 else 1
    new_team = pd.DataFrame({
        'team_id': [new_id],
        'name': [name]
    })
    teams_df = pd.concat([teams_df, new_team], ignore_index=True)
    return new_id, teams_df

In [4]:
def scrape_dual_page(url):
    """Scrape a single dual meet page"""
    r = requests.get(url)
    s = BeautifulSoup(r.text, 'html.parser')
    
    # Extract table
    table = s.find('table', class_='bg-lightdark table table-hover table-sm')
    if not table:
        return None, None
    
    headers = [th.get_text(strip=True) for th in table.find_all('th')]
    rows = []
    for tr in table.find_all('tr')[1:]:
        cells = [td.get_text(strip=True) for td in tr.find_all('td')]
        if cells:
            rows.append(cells)
    
    df = pd.DataFrame(rows, columns=headers)
    
    # Extract footer
    footer = s.find('div', class_='card-footer')
    footer_text = footer.get_text(strip=True) if footer else "Unknown"
    
    return df, footer_text

In [5]:
def parse_dual_footer(footer_text):
    """Parse dual meet metadata from footer text"""
    # Pattern: #32 Purdue over #47 Cal Poly 19 - 13 ... event date: 11/16/25
    pattern = (
        r'#(\d+)\s+([A-Za-z0-9\-\.\'&\s]+?)\s+over\s+#(\d+)\s+([A-Za-z0-9\-\.\'&\s]+?)\s+'
        r'(\d+)\s*-\s*(\d+).*?event date:\s*(\d{2}/\d{2}/\d{2})'
    )
    
    match = re.search(pattern, footer_text)
    if not match:
        return None
    
    return {
        'team1_rank': int(match.group(1)),
        'team1_name': match.group(2).strip(),
        'team2_rank': int(match.group(3)),
        'team2_name': match.group(4).strip(),
        'team1_score': int(match.group(5)),
        'team2_score': int(match.group(6)),
        'event_date': match.group(7)
    }

def process_dual_data(df, footer_info, wrestlers_df, teams_df, dual_id):
    """Clean and process a single dual meet's data"""
    
    # Parse table header for team names
    col_names = df.columns.tolist()
    if len(col_names) < 3:
        return None, wrestlers_df, teams_df
    
    table_team1 = col_names[1].replace("Wrestler", "").strip()
    table_team2 = col_names[2].replace("Wrestler", "").strip()
    
    # Best match logic for footer alignment
    def best_match(team_name, candidates):
        team_lower = team_name.lower()
        for candidate in candidates:
            if candidate.lower() in team_lower or team_lower in candidate.lower():
                return candidate
        return max(candidates, key=lambda c: len(set(c.lower()) & set(team_lower)))
    
    footer_teams = [footer_info['team1_name'], footer_info['team2_name']]
    team1_footer = best_match(table_team1, footer_teams)
    team2_footer = best_match(table_team2, footer_teams)
    
    # Determine which is team1 (winner) vs team2
    is_team1_winner = team1_footer == footer_info['team1_name']
    
    if is_team1_winner:
        home_team_name = footer_info['team1_name']
        home_team_rank = footer_info['team1_rank']
        away_team_name = footer_info['team2_name']
        away_team_rank = footer_info['team2_rank']
    else:
        home_team_name = footer_info['team2_name']
        home_team_rank = footer_info['team2_rank']
        away_team_name = footer_info['team1_name']
        away_team_rank = footer_info['team1_rank']
    
    # Get or create team IDs
    home_team_id, teams_df = get_or_create_team(teams_df, home_team_name)
    away_team_id, teams_df = get_or_create_team(teams_df, away_team_name)
    
    # Extract wrestler info
    wrestler_pattern = r'#\s*(\d+)\s*(.+)'
    df[['home_rank', 'home_name']] = df[col_names[1]].str.extract(wrestler_pattern)
    df[['away_rank', 'away_name']] = df[col_names[2]].str.extract(wrestler_pattern)
    
    # Get or create wrestler IDs
    home_wrestler_ids = []
    away_wrestler_ids = []
    
    for name in df['home_name']:
        wrestler_id, wrestlers_df = get_or_create_wrestler(wrestlers_df, name)
        home_wrestler_ids.append(wrestler_id)
    
    for name in df['away_name']:
        wrestler_id, wrestlers_df = get_or_create_wrestler(wrestlers_df, name)
        away_wrestler_ids.append(wrestler_id)
    
    df['home_wrestler_id'] = home_wrestler_ids
    df['away_wrestler_id'] = away_wrestler_ids
    
    # Parse results
    df['home_rank'] = pd.to_numeric(df['home_rank'], errors='coerce').astype('Int64')
    df['away_rank'] = pd.to_numeric(df['away_rank'], errors='coerce').astype('Int64')
    df['home_win'] = df['Result'].str.startswith('W')
    df['win_type'] = df['Result'].str.extract(r'([A-Z]{2,}|TB-\d|INJ|FOR)')[0].fillna("UNK")
    
    # Add metadata
    df['dual_id'] = dual_id
    df['home_team_id'] = home_team_id
    df['away_team_id'] = away_team_id
    df['event_date'] = footer_info['event_date']
    df['weight_class'] = df['Weight']
    
    # Select final columns
    matches_df = df[[
        'dual_id', 'weight_class', 'event_date',
        'home_team_id', 'home_wrestler_id', 'home_name', 'home_rank',
        'away_team_id', 'away_wrestler_id', 'away_name', 'away_rank',
        'home_win', 'win_type', 'Result'
    ]].copy()
    
    return matches_df, wrestlers_df, teams_df

In [6]:
def main():
    print("🤼 Starting Wrestling Data Collection Pipeline")
    print("=" * 60)
    
    # Load existing data
    if LINKS_FILE.exists():
        with open(LINKS_FILE, "rb") as f:
            seen_links = pickle.load(f)
    else:
        seen_links = set()
    
    wrestlers_df = load_or_create_df(WRESTLERS_FILE, ['wrestler_id', 'name'])
    teams_df = load_or_create_df(TEAMS_FILE, ['team_id', 'name'])
    dual_meets_df = load_or_create_df(DUAL_MEETS_FILE, [
        'dual_id', 'team1_id', 'team1_rank', 'team1_score',
        'team2_id', 'team2_rank', 'team2_score', 'event_date'
    ])
    matches_df = load_or_create_df(MATCHES_FILE, [
        'dual_id', 'weight_class', 'event_date',
        'home_team_id', 'home_wrestler_id', 'home_name', 'home_rank',
        'away_team_id', 'away_wrestler_id', 'away_name', 'away_rank',
        'home_win', 'win_type', 'Result'
    ])
    
    print(f"📊 Loaded existing data:")
    print(f"   • Wrestlers: {len(wrestlers_df)}")
    print(f"   • Teams: {len(teams_df)}")
    print(f"   • Dual Meets: {len(dual_meets_df)}")
    print(f"   • Matches: {len(matches_df)}")
    print(f"   • Seen Links: {len(seen_links)}\n")
    
    # Scrape new duals
    response = requests.get(DUALS_URL)
    soup = BeautifulSoup(response.text, "html.parser")
    dual_divs = soup.find_all("div", class_="col-3 col-sm-2 my-auto text-right")
    dual_links = [BASE_URL + a.find("a")["href"] for a in dual_divs if a.find("a")]
    
    new_links = [link for link in dual_links if link not in seen_links]
    print(f"🆕 Found {len(new_links)} new dual(s) to scrape\n")
    
    if not new_links:
        print("✅ No new duals found. Data is up to date!")
        return
    
    # Process new duals
    new_matches = []
    next_dual_id = dual_meets_df['dual_id'].max() + 1 if len(dual_meets_df) > 0 else 1
    
    for link in new_links:
        try:
            print(f"📥 Scraping: {link}")
            
            # Scrape page
            table_df, footer_text = scrape_dual_page(link)
            if table_df is None:
                print(f"   ⏭️  No table found, skipping")
                continue
            
            # Parse footer
            footer_info = parse_dual_footer(footer_text)
            if not footer_info:
                print(f"   ⏭️  Unranked or malformed dual, skipping")
                continue
            
            print(f"   ✓ #{footer_info['team1_rank']} {footer_info['team1_name']} over "
                  f"#{footer_info['team2_rank']} {footer_info['team2_name']} "
                  f"({footer_info['team1_score']}-{footer_info['team2_score']})")
            
            # Process matches
            matches, wrestlers_df, teams_df = process_dual_data(
                table_df, footer_info, wrestlers_df, teams_df, next_dual_id
            )
            
            if matches is None:
                continue
            
            # Get team IDs for dual meet record
            team1_id, teams_df = get_or_create_team(teams_df, footer_info['team1_name'])
            team2_id, teams_df = get_or_create_team(teams_df, footer_info['team2_name'])
            
            # Add dual meet record
            new_dual = pd.DataFrame({
                'dual_id': [next_dual_id],
                'team1_id': [team1_id],
                'team1_rank': [footer_info['team1_rank']],
                'team1_score': [footer_info['team1_score']],
                'team2_id': [team2_id],
                'team2_rank': [footer_info['team2_rank']],
                'team2_score': [footer_info['team2_score']],
                'event_date': [footer_info['event_date']]
            })
            dual_meets_df = pd.concat([dual_meets_df, new_dual], ignore_index=True)
            
            new_matches.append(matches)
            seen_links.add(link)
            next_dual_id += 1
            
            time.sleep(1)  # Be polite to server
            
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
            continue
    
    # Combine and save
    if new_matches:
        new_matches_df = pd.concat(new_matches, ignore_index=True)
        matches_df = pd.concat([matches_df, new_matches_df], ignore_index=True)
        
        # Save all files
        wrestlers_df.to_csv(WRESTLERS_FILE, index=False)
        teams_df.to_csv(TEAMS_FILE, index=False)
        dual_meets_df.to_csv(DUAL_MEETS_FILE, index=False)
        matches_df.to_csv(MATCHES_FILE, index=False)
        
        with open(LINKS_FILE, "wb") as f:
            pickle.dump(seen_links, f)
        
        print("\n" + "=" * 60)
        print("✅ Data Collection Complete!")
        print(f"   • Added {len(new_matches_df)} new matches")
        print(f"   • Added {len(dual_meets_df) - (len(dual_meets_df) - len(new_matches))} new duals")
        print(f"\n📊 Current Totals:")
        print(f"   • Wrestlers: {len(wrestlers_df)}")
        print(f"   • Teams: {len(teams_df)}")
        print(f"   • Dual Meets: {len(dual_meets_df)}")
        print(f"   • Matches: {len(matches_df)}")
    else:
        print("\n✅ No new ranked D1 duals found this week")

In [7]:
if __name__ == "__main__":
    main()

🤼 Starting Wrestling Data Collection Pipeline
📊 Loaded existing data:
   • Wrestlers: 1512
   • Teams: 78
   • Dual Meets: 584
   • Matches: 5840
   • Seen Links: 584

🆕 Found 53 new dual(s) to scrape

📥 Scraping: https://www.wrestlestat.com/event/96330/bloomsburg-mercyhurst-dual/boxscore
   ⏭️  Unranked or malformed dual, skipping
📥 Scraping: https://www.wrestlestat.com/event/96944/mercyhurst-edinboro-dual/boxscore
   ⏭️  Unranked or malformed dual, skipping
📥 Scraping: https://www.wrestlestat.com/event/96144/franklin-and-marshall-millersville-dual/boxscore
   ⏭️  Unranked or malformed dual, skipping
📥 Scraping: https://www.wrestlestat.com/event/106303/mercyhurst-kent-state-dual/boxscore
   ⏭️  Unranked or malformed dual, skipping
📥 Scraping: https://www.wrestlestat.com/event/97455/westcliff-csu-bakersfield-dual/boxscore
   ⏭️  Unranked or malformed dual, skipping
📥 Scraping: https://www.wrestlestat.com/event/97427/penn-state-behrend-mercyhurst-dual/boxscore
   ⏭️  Unranked or malform

In [8]:
# File paths
DATA_DIR = Path("wrestling_data")
WRESTLERS_FILE = DATA_DIR / "wrestlers.csv"
TEAMS_FILE = DATA_DIR / "teams.csv"
DUAL_MEETS_FILE = DATA_DIR / "dual_meets.csv"
MATCHES_FILE = DATA_DIR / "matches.csv"

# Load all dataframes
wrestlers_df = pd.read_csv(WRESTLERS_FILE)
teams_df = pd.read_csv(TEAMS_FILE)
dual_meets_df = pd.read_csv(DUAL_MEETS_FILE)
matches_df = pd.read_csv(MATCHES_FILE)

# Display each dataframe
print("=" * 80)
print("WRESTLERS")
print("=" * 80)
print(f"Shape: {wrestlers_df.shape}\n")
print("HEAD:")
display(wrestlers_df.head())
print("\nTAIL:")
display(wrestlers_df.tail())

print("\n" + "=" * 80)
print("TEAMS")
print("=" * 80)
print(f"Shape: {teams_df.shape}\n")
print("HEAD:")
display(teams_df.head())
print("\nTAIL:")
display(teams_df.tail())

print("\n" + "=" * 80)
print("DUAL MEETS")
print("=" * 80)
print(f"Shape: {dual_meets_df.shape}\n")
print("HEAD:")
display(dual_meets_df.head())
print("\nTAIL:")
display(dual_meets_df.tail())

print("\n" + "=" * 80)
print("MATCHES")
print("=" * 80)
print(f"Shape: {matches_df.shape}\n")
print("HEAD:")
display(matches_df.head())
print("\nTAIL:")
display(matches_df.tail())

WRESTLERS
Shape: (1512, 2)

HEAD:


,wrestler_id,name
0,1,Bowen Downey
1,2,Julian Farber
2,3,Max Brady
3,4,Ethan Basile
4,5,Cael Rahnavardi



TAIL:


,wrestler_id,name
1507,1508,Nieko Malone
1508,1509,Max Kirby
1509,1510,Till Helms
1510,1511,Jackson Lusk
1511,1512,Matthew Rogers



TEAMS
Shape: (78, 2)

HEAD:


...

,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
579,580,50,26,22,33,35,18,02/19/26
580,581,63,47,28,8,75,10,02/19/26
581,582,44,39,37,34,57,3,02/19/26
582,583,54,42,32,60,65,14,02/19/26
583,584,12,8,45,43,68,3,02/18/26



MATCHES
Shape: (5840, 14)

HEAD:


,dual_id,weight_class,event_date,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result
0,1,125,01/10/26,1,1.0,Bowen Downey,62.0,2,11.0,Brady Roark,42.0,False,LDEC,LDEC 6 - 2
1,1,133,01/10/26,1,2.0,Julian Farber,33.0,2,12.0,Cale Seaton,43.0,True,WDEC,WDEC 12 - 6
2,1,141,01/10/26,1,3.0,Max Brady,74.0,2,13.0,Julian Tagg,19.0,True,WDEC,WDEC 12 - 10
3,1,149,01/10/26,1,4.0,Ethan Basile,111.0,2,14.0,Logan Swensen,83.0,True,WTF,WTF5 15 - 0 7:00
4,1,157,01/10/26,1,5.0,Cael Rahnavardi,105.0,2,15.0,Cael Swensen,20.0,False,LTF,LTF5 19 - 4 7:00



TAIL:


,dual_id,weight_class,event_date,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result
5835,584,165,02/18/26,43,467.0,Jake Slotnick,76.0,12,938.0,Andrew Barbosa,17.0,False,LDEC,LDEC 4 - 1
5836,584,174,02/18/26,43,1512.0,Matthew Rogers,225.0,12,939.0,Lenny Pinto,28.0,False,LTF,LTF5 18 - 3 5:58
5837,584,184,02/18/26,43,577.0,Josh Jorgge,174.0,12,135.0,Jordan Chapman,32.0,False,LFALL,LFALL 2:06
5838,584,197,02/18/26,43,470.0,Will Conlon,81.0,12,1027.0,PJ Casale,64.0,False,LINJ,LINJ 4:43
5839,584,285,02/18/26,43,471.0,Adrian Sans,183.0,12,138.0,Hunter Catka,23.0,False,LFALL,LFALL 2:31


In [9]:
# -- Save data and set aside
matches_df_temp = matches_df.copy(deep=True)

In [10]:
# -- Manipulate
matches_df_temp['event_date'] = pd.to_datetime(matches_df_temp['event_date'])
matches_df_temp['home_won'] = matches_df_temp['Result'].str.startswith('W') 


# -- Join team data
matches_df_temp = matches_df_temp.merge(
    teams_df.rename(columns={'name': 'home_team_name'}),
    how='left',
    left_on='home_team_id',
    right_on='team_id'
)

# Merge away team name
matches_df_temp = matches_df_temp.merge(
    teams_df.rename(columns={'name': 'away_team_name'}),
    how='left',
    left_on='away_team_id',
    right_on='team_id'
)

# Drop duplicate team_id columns
matches_df_temp = matches_df_temp.drop(columns=['team_id_x', 'team_id_y'])
matches_df_temp = matches_df_temp.set_index('event_date')
matches_df_temp = matches_df_temp.sort_index()

matches_df_temp

,dual_id,weight_class,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_won,home_team_name,away_team_name
event_date,,,,,,,,,,,,,,,,
2025-11-01,280,141,24,267.0,Raymond Adams,160.0,67,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,False,Duke,Sacred Heart
2025-11-01,283,197,70,878.0,Kael Bennie,46.0,23,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,False,Utah Valley,Stanford
2025-11-01,283,285,70,879.0,Jack Forbes,62.0,23,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,True,Utah Valley,Stanford
2025-11-01,284,125,64,784.0,Koda Holeman,40.0,20,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,True,Cal Poly,Purdue
2025-11-01,284,133,64,785.0,Anthony Lucio,245.0,20,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,False,Cal Poly,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,547,174,37,399.0,Joshua Roe,237.0,66,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,False,Presbyterian,VMI
2026-02-23,547,184,37,400.0,Reed Douglass,72.0,66,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,True,Presbyterian,VMI
2026-02-23,547,197,37,911.0,Toler Hornick,207.0,66,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,False,Presbyterian,VMI


In [11]:
# # # -- November and December data
# nov = matches_df_temp['2025-11':'2025-11']
# nov.to_csv("November_Matches.csv")

# dec = matches_df_temp['2025-12':'2025-12']
# dec.to_csv("December_Matches.csv")

# jan = matches_df_temp['2026-1':'2026-1']
# jan.to_csv('January_Matches.csv')


feb = matches_df_temp['2026-2':'2026-2']
feb.to_csv('Febuary_Matches.csv')

# print("="*60, " November Dual Matches", "="*60)
# display(nov)
# print()
# print("="*60, " December Dual Matches", "="*60)
# display(dec)
# print("="*60, " January Dual Matches", "="*60)
# display(jan

print("="*60, "Febuary Dual Matches", "="*60)
display(feb)

============================================================ Febuary Dual Matches ============================================================


,dual_id,weight_class,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_won,home_team_name,away_team_name
event_date,,,,,,,,,,,,,,,,
2026-02-01,537,285,75,967.0,Brady Colbert,15.0,15,168.0,Nathan Taylor,6.0,True,WDEC,WDEC 8 - 1,True,Army West Point,Lehigh
2026-02-01,537,197,75,1052.0,Wolfgang Frable,39.0,15,167.0,JT Davis,29.0,False,LSV,LSV-1 4 - 1,False,Army West Point,Lehigh
2026-02-01,537,184,75,965.0,David Barrett,85.0,15,166.0,Rylan Rogers,28.0,False,LMD,LMD 15 - 4,False,Army West Point,Lehigh
2026-02-01,537,174,75,964.0,Cooper Haase,64.0,15,1175.0,Zeke Dubler,109.0,True,WMD,WMD 16 - 2,True,Army West Point,Lehigh
2026-02-01,537,165,75,1051.0,Gunner Filipowicz,26.0,15,164.0,Max Brignola,6.0,False,LFALL,LFALL 5:36,False,Army West Point,Lehigh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,547,174,37,399.0,Joshua Roe,237.0,66,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,False,Presbyterian,VMI
2026-02-23,547,184,37,400.0,Reed Douglass,72.0,66,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,True,Presbyterian,VMI
2026-02-23,547,197,37,911.0,Toler Hornick,207.0,66,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,False,Presbyterian,VMI


In [12]:
# Query for wrestler
def query_wrestler(wrestler_name: str, df):
    display(df.query('home_name == @wrestler_name or away_name == @wrestler_name'))

query_wrestler("Richard Castro-Sandoval", matches_df_temp)

,dual_id,weight_class,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_won,home_team_name,away_team_name
event_date,,,,,,,,,,,,,,,,
2025-11-02,279,125,10,102.0,Richard Castro-Sandoval,71.0,20,1105.0,Isaiah Quintero,64.0,True,WDEC,WDEC 2 - 0,True,CSU Bakersfield,Purdue
2025-11-14,248,125,64,784.0,Koda Holeman,40.0,10,102.0,Richard Castro-Sandoval,71.0,True,WMD,WMD 11 - 3,True,Cal Poly,CSU Bakersfield
2025-12-21,81,125,5,721.0,Colton Camacho,20.0,10,102.0,Richard Castro-Sandoval,71.0,False,LINJ,LINJ 7:00,False,Central Michigan,CSU Bakersfield
2025-12-21,85,125,10,102.0,Richard Castro-Sandoval,71.0,59,711.0,JJ Peace,54.0,False,LDEC,LDEC 4 - 1,False,CSU Bakersfield,American
2026-01-09,26,125,10,102.0,Richard Castro-Sandoval,71.0,1,31.0,Trever Anderson,21.0,False,LMD,LMD 11 - 3,False,CSU Bakersfield,Northern Iowa
2026-01-09,27,125,10,102.0,Richard Castro-Sandoval,71.0,6,58.0,Brayden Teunissen,91.0,True,WDEC,WDEC 6 - 1,True,CSU Bakersfield,Ohio
2026-01-09,28,125,10,102.0,Richard Castro-Sandoval,71.0,8,81.0,Gavin Martin,199.0,True,WTF,WTF5 20 - 5 3:49,True,CSU Bakersfield,Bloomsburg
2026-01-10,10,125,10,102.0,Richard Castro-Sandoval,71.0,3,112.0,Oumar Tounkara,63.0,True,WDEC,WDEC 8 - 7,True,CSU Bakersfield,Drexel
2026-01-10,11,125,10,102.0,Richard Castro-Sandoval,71.0,7,68.0,Logan Brzozowski,75.0,False,LDEC,LDEC 4 - 2,False,CSU Bakersfield,Harvard


In [13]:
dual_meets_df_temp = dual_meets_df.copy(deep=True)

In [14]:
# -- Save dual data November & December
dual_meets_df_temp['event_date'] = pd.to_datetime(dual_meets_df_temp['event_date'])
dual_meets_df_temp['team1_won'] = dual_meets_df_temp['team1_score'] > dual_meets_df_temp['team2_score'] 

# -- Join team data
dual_meets_df_temp = dual_meets_df_temp.merge(
    teams_df.rename(columns={'name': 'team1_name'}),
    how='left',
    left_on='team1_id',
    right_on='team_id'
)

# Merge away team name
dual_meets_df_temp = dual_meets_df_temp.merge(
    teams_df.rename(columns={'name': 'team2_name'}),
    how='left',
    left_on='team2_id',
    right_on='team_id'
)


dual_meets_df_temp = dual_meets_df_temp.drop(columns=['team_id_x', 'team_id_y'])
dual_meets_df_temp = dual_meets_df_temp.set_index('event_date')
dual_meets_df_temp = dual_meets_df_temp.sort_index()

In [15]:
# #-- Save
# nov_dual = dual_meets_df_temp['2025-11':'2025-11']
# nov_dual.to_csv('November_Duals.csv')

# dec_dual = dual_meets_df_temp['2025-12':'2025-12']
# dec_dual.to_csv('December_Duals.csv')

# jan_dual = dual_meets_df_temp['2026-1':'2026-1']
# jan_dual.to_csv('January_Duals.csv')

feb_dual = dual_meets_df_temp['2026-2':'2026-2']
feb_dual.to_csv("Febuary_Duals.csv")

# print("="*60, "November Duals", "="*60)
# display(nov_dual)
# print()
# print("="*60, "December Duals", "="*60)
# display(dec_dual)
# print("="*60, "January Duals", "="*60)
# display(jan_dual)

print("="*60, "Febuary Duals", "="*60)
display(feb_dual)

============================================================ Febuary Duals ============================================================


,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,team1_won,team1_name,team2_name
event_date,,,,,,,,,,
2026-02-01,533,41,2,26,25,9,16,True,Ohio State,Michigan
2026-02-01,534,52,3,24,39,4,9,True,Oklahoma State,Iowa State
2026-02-01,535,2,20,31,49,63,10,True,South Dakota State,Northern Colorado
2026-02-01,536,62,29,23,71,51,15,True,North Dakota State,Air Force
2026-02-01,538,44,36,27,54,48,14,True,Lock Haven,George Mason
...,...,...,...,...,...,...,...,...,...,...
2026-02-22,551,39,4,20,73,14,14,True,Iowa State,Missouri
2026-02-22,550,52,2,32,17,5,11,True,Oklahoma State,Iowa
2026-02-22,549,25,7,34,5,43,6,True,Michigan,Central Michigan


In [16]:
	#68 Castro-Sandoval, Richard 

s = "#68 Castro-Sandoval, Richard"

# Remove the ranking prefix
clean = s.split(" ", 1)[1]          # "Owens, Tucker"

# Split on comma and reorder
last, first = [x.strip() for x in clean.split(",")]
result = f"{first} {last}"

print(result)


Richard Castro-Sandoval


In [17]:
matches_df_temp

,dual_id,weight_class,home_team_id,home_wrestler_id,home_name,home_rank,away_team_id,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_won,home_team_name,away_team_name
event_date,,,,,,,,,,,,,,,,
2025-11-01,280,141,24,267.0,Raymond Adams,160.0,67,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,False,Duke,Sacred Heart
2025-11-01,283,197,70,878.0,Kael Bennie,46.0,23,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,False,Utah Valley,Stanford
2025-11-01,283,285,70,879.0,Jack Forbes,62.0,23,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,True,Utah Valley,Stanford
2025-11-01,284,125,64,784.0,Koda Holeman,40.0,20,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,True,Cal Poly,Purdue
2025-11-01,284,133,64,785.0,Anthony Lucio,245.0,20,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,False,Cal Poly,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,547,174,37,399.0,Joshua Roe,237.0,66,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,False,Presbyterian,VMI
2026-02-23,547,184,37,400.0,Reed Douglass,72.0,66,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,True,Presbyterian,VMI
2026-02-23,547,197,37,911.0,Toler Hornick,207.0,66,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,False,Presbyterian,VMI
